# Reproduce the film chart for cutoff 8

Restart the Python kernel and run the cells in order. The six films are synthetic teaching data. The notebook embeds the CSV and checks its SHA-256. A film can meet the rating cutoff and lack a revenue value. Each chart point represents one film with both values.

Use the Chart Reproducibility Lab's pinned Python requirements. The final cell saves a PNG and SVG to the current working directory.

In [ ]:
from hashlib import sha256
from importlib.metadata import version
from io import StringIO
import json
import platform
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

matplotlib.rcParams.update({
    "font.family": "DejaVu Sans",
    "svg.fonttype": "none",
    "svg.hashsalt": "dvj-chart-reproducibility-v1",
})
runtime = {
    "python": platform.python_version(),
    "implementation": platform.python_implementation(),
    "kernel": "ipykernel",
    "packages": {name: version(name) for name in (
        "ipykernel", "jupyter-client", "matplotlib", "nbclient", "nbformat",
        "numpy", "pandas", "Pillow",
    )},
}
print("DVJ_RUNTIME " + json.dumps(runtime))


In [ ]:
minimum_rating = 8
print("DVJ_PARAMETER " + json.dumps({"requestedCutoff": minimum_rating}))


In [ ]:
# This embedded copy lets the notebook run on its own.
source_csv = 'film_id,rating,revenue_millions\nF1,5,8\nF2,6,12\nF3,7,18\nF4,8,22\nF5,9,30\nF6,8,\n'
expected_sha256 = 'b04401595bf87b35e46cada9d197415919446845c747afa65732cbb5c360b1d3'
source_sha256 = sha256(source_csv.encode("utf-8")).hexdigest()
if source_sha256 != expected_sha256:
    raise ValueError("The embedded source differs from the published fixture.")

films = pd.read_csv(StringIO(source_csv), dtype={
    "film_id": "string", "rating": "Int64", "revenue_millions": "Int64",
})
if list(films.columns) != ["film_id", "rating", "revenue_millions"]:
    raise ValueError("Unexpected source columns.")
if len(films) != 6 or not films.film_id.is_unique or films.rating.isna().any():
    raise ValueError("Unexpected fixture count, identifier, or rating.")
print("DVJ_SOURCE " + json.dumps({"rows": len(films), "sha256": source_sha256}))
print(films.to_string(index=False))


In [ ]:
analysis_cutoff = minimum_rating
eligible = films.loc[films.rating.ge(analysis_cutoff)]
plotted = eligible.dropna(subset=["revenue_millions"])
summary = {
    "eligibleIds": eligible.film_id.tolist(),
    "plottedIds": plotted.film_id.tolist(),
    "missingIds": eligible.loc[eligible.revenue_millions.isna(), "film_id"].tolist(),
    "revenueSumUsdMillions": int(plotted.revenue_millions.sum()),
    "points": [
        {"id": row.film_id, "rating": int(row.rating),
         "revenueMillions": int(row.revenue_millions)}
        for row in plotted.itertuples(index=False)
    ],
}
print("DVJ_SUMMARY " + json.dumps({"analysisCutoff": analysis_cutoff, "summary": summary}))
print(f"{len(eligible)} eligible. {len(plotted)} plotted. "
      f"{len(summary['missingIds'])} eligible with missing revenue.")


In [ ]:
title = f"Films with rating at least {minimum_rating}"
revision = f"cutoff-{minimum_rating}"
figure = plt.figure(figsize=(6, 4), dpi=100, facecolor="white")
axes = figure.add_axes([.16, .17, .78, .71])
axes.set(
    title=title, xlabel="Rating (out of 10)", ylabel="Revenue (USD millions)",
    xlim=(4, 10), ylim=(0, 36), xticks=range(4, 11), yticks=range(0, 37, 6),
)
axes.title.set_gid("artifact-title")
axes.grid(axis="y", color=".90")
for row in plotted.itertuples(index=False):
    glyph = axes.scatter(int(row.rating), int(row.revenue_millions),
                         s=88, color="#633897", edgecolors="none", zorder=3)
    glyph.set_gid(f"film-{row.film_id}")

figure.canvas.draw()
extent = axes.get_window_extent()
_, image_height = figure.canvas.get_width_height()
plot_bounds = {
    "left": float(extent.x0), "right": float(extent.x1),
    "top": float(image_height - extent.y1), "bottom": float(image_height - extent.y0),
}
figure.savefig("rating-revenue.png", metadata={"Title": title, "Revision": revision})
figure.savefig("rating-revenue.svg", metadata={
    "Title": title, "Description": revision, "Date": None,
})
plt.close(figure)
print("DVJ_ARTIFACT " + json.dumps({
    "revision": revision, "title": title,
    "xLabel": axes.get_xlabel(), "yLabel": axes.get_ylabel(),
    "xDomain": list(axes.get_xlim()), "yDomain": list(axes.get_ylim()), "plotBounds": plot_bounds,
}))
print("Saved rating-revenue.png and rating-revenue.svg.")
